# 🧠 StudySync — Ollama AI Backend

Sets up a local Ollama server and exposes it publicly via **ngrok** so StudySync can use it as the AI backend.

### ✅ How to Use
1. Run **Cell 1** — pick your model
2. Run **Cell 2** — start the server
3. Copy the public URL → paste into your StudySync `backend/.env` as `OLLAMA_BASE_URL`

> ⚠️ Go to `Runtime > Change runtime type` and select **GPU (T4)** before running.

In [ ]:
#@title 📋 Model Registry
# @markdown ### Model Configuration
# @markdown > Enter the model names you want to add, separated by commas. Guide: `8B` recommended · `14B` feasible · `20B+` slow
# @markdown > - Find official model names here: https://ollama.com/search
model_list = "mistral, llama3.2:3b, llama3.1:8b, phi4-mini, phi3" #@param {type:"string"}
# @markdown ### Context Length (num_ctx)
# @markdown > T4 (16GB VRAM) Guide
# @markdown > | Model Size | Rec. ctx | Notes |
# @markdown > |---|---|---|
# @markdown > | ~4B  | 32768 | Plentiful |
# @markdown > | ~8B  | 16384 | Standard |
# @markdown > | ~14B | 8192  | VRAM tight |
# @markdown > | ~20B+ | 4096 | Min. usage |
# @markdown >
# @markdown > Default (0) lets Ollama decide (equivalent to 4096).
num_ctx = 16384 #@param {type:"integer"}

AVAILABLE_MODELS = [
    model.strip()
    for model in model_list.split(',')
    if model.strip()
]

if not AVAILABLE_MODELS:
    raise ValueError("❌ Model list is empty. Please enter at least one model.")

import ipywidgets as widgets
from IPython.display import display

header = widgets.HTML(
    '<h3>📦 Select a Model</h3>'
    '<p style="margin: 5px 0 10px 0; font-size: 13px;">'
    'Select one model to launch, then run the next cell.</p>'
)

model_selector = widgets.RadioButtons(
    options=AVAILABLE_MODELS,
    value=AVAILABLE_MODELS[0],
    layout=widgets.Layout(padding='0 0 0 20px')
)

display(widgets.VBox([header, model_selector]))
print(f"✅ Model list loaded: {len(AVAILABLE_MODELS)} model(s)")
print("➡️  After selecting a model, run the next cell (Server).")

In [ ]:
#@title 🚀 Ollama Colab Server — StudySync Edition

# @markdown > Create a free Ngrok account at the URL below and paste your auth token here.
# @markdown > - https://dashboard.ngrok.com/get-started/your-authtoken

ngrok_token = "" #@param {type:"string"}

MAX_HEALTH_RETRIES   = 150
HEALTH_CHECK_TIMEOUT = 2
STATUS_POLL_INTERVAL = 30

BLUE  = "\033[34m"
GREEN = "\033[32m"
GRAY  = "\033[90m"
RESET = "\033[0m"

selected_model = model_selector.value
effective_ctx  = num_ctx if num_ctx > 0 else None

print(f"\nSTUDYSYNC OLLAMA SERVER 🚀")
print(f"{GRAY}──────────────────────────────────────────{RESET}")

print(f"  {BLUE}◦{RESET} System   Installing dependencies (zstd)")
!which zstd > /dev/null 2>&1 || (apt-get update -qq && apt-get install -y -qq zstd > /dev/null)

print(f"  {BLUE}◦{RESET} System   Installing Ollama")
!which ollama > /dev/null 2>&1 || curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

print(f"  {BLUE}◦{RESET} System   Installing pyngrok")
!python -c "import pyngrok" > /dev/null 2>&1 || pip install -q pyngrok

import re
import subprocess
import time
import os
import requests

if not re.fullmatch(r'[a-zA-Z0-9._:/-]+', selected_model):
    raise ValueError(f"Invalid model name: {selected_model}")

os.environ['OLLAMA_HOST']              = '0.0.0.0:11434'
os.environ['OLLAMA_KEEP_ALIVE']        = '24h'
os.environ['OLLAMA_MAX_LOADED_MODELS'] = '1'
os.environ['OLLAMA_FLASH_ATTENTION']   = '1'
os.environ['OLLAMA_KV_CACHE_TYPE']     = 'q8_0'

process = subprocess.Popen(
    ["/usr/local/bin/ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

for _ in range(MAX_HEALTH_RETRIES):
    try:
        if requests.get("http://0.0.0.0:11434/api/tags", timeout=HEALTH_CHECK_TIMEOUT).status_code == 200:
            print(f"  {GREEN}✓{RESET} Ready    Ollama server started (0.0.0.0:11434)")
            break
    except requests.exceptions.RequestException:
        pass
    time.sleep(0.2)
else:
    raise RuntimeError("⚠️ Failed to confirm Ollama server startup.")

print(f"  {BLUE}◦{RESET} Network  Establishing ngrok tunnel (port: 11434)")
from pyngrok import ngrok

if not ngrok_token:
    raise ValueError("⚠️ ngrok token is not set. Paste your token in the field above.")

ngrok.set_auth_token(ngrok_token)
ngrok.kill()
tunnel = ngrok.connect(11434)
public_url = tunnel.public_url

print(f"  {GREEN}✓{RESET} Public   {public_url}")
print(f"\n  ▶ Model    {selected_model}")
ctx_label = str(effective_ctx) if effective_ctx else "Ollama Default (4096)"
print(f"  {GRAY}  num_ctx  {ctx_label}{RESET}")
print(f"  {GRAY}└ Pulling  Download started... (first time: ~5–15 min){RESET}")

subprocess.run(
    ["/usr/local/bin/ollama", "pull", selected_model],
    env={**os.environ, "OLLAMA_HOST": "0.0.0.0:11434"},
    check=True
)
print(f"  {GREEN}✓{RESET} Loaded   Model loaded successfully")

warmup_payload = {
    "model": selected_model,
    "prompt": "hi",
    "stream": False,
}
if effective_ctx:
    warmup_payload["options"] = {"num_ctx": effective_ctx}

try:
    requests.post(
        "http://0.0.0.0:11434/api/generate",
        json=warmup_payload,
        timeout=180
    )
    ctx_info = f" (num_ctx: {effective_ctx})" if effective_ctx else ""
    print(f"  {GREEN}✓{RESET} Warmed   Model loaded into VRAM{ctx_info}")
except requests.exceptions.RequestException as e:
    print(f"  ⚠️  Warmup   Warm-up failed: {e}")

ctx_disp = str(effective_ctx) if effective_ctx else "4096 (Ollama Default)"
print(f"\n{GRAY}──────────────────────────────────────────{RESET}")
print(f"ENDPOINT : {public_url}")
print(f"NUM_CTX  : {ctx_disp}")
print(f"{GRAY}──────────────────────────────────────────{RESET}")

print(f"\n📋 StudySync .env config")
print(f"{GRAY}Paste into your backend/.env:{RESET}")
print(f"""
OLLAMA_BASE_URL={public_url}
OLLAMA_MODEL={selected_model}
""")

print(f"📋 services.js snippet")
print(f"{GRAY}Replace the Gemini call in backend/services/services.js:{RESET}")
print('''
const axios = require('axios');

async function aiChat(userMessage) {
  const response = await axios.post(`${process.env.OLLAMA_BASE_URL}/api/chat`, {
    model: process.env.OLLAMA_MODEL || 'mistral',
    messages: [{ role: 'user', content: userMessage }],
    stream: false
  });
  return response.data.message.content;
}

module.exports = { aiChat };
''')

try:
    start_time = time.time()
    while True:
        elapsed_min = int((time.time() - start_time) / 60)
        print(f"\r  {GREEN}●{RESET} Running  Uptime: {elapsed_min}min | {public_url}", end="")
        time.sleep(STATUS_POLL_INTERVAL)
except KeyboardInterrupt:
    print(f"\n\n  {GRAY}Session terminated.{RESET}")
finally:
    ngrok.disconnect(public_url)

In [ ]:
#@title 🧪 Test the Model (Optional)
# @markdown Sends a quick message to confirm the model is responding.

import requests

test_response = requests.post(
    'http://0.0.0.0:11434/api/chat',
    json={
        'model': model_selector.value,
        'messages': [{'role': 'user', 'content': 'Give me one quick study tip in one sentence.'}],
        'stream': False
    }
)

result = test_response.json()
print('🤖 Model response:')
print(result['message']['content'])

In [ ]:
#@title 🛑 Stop Server
# @markdown Run this when you're done to cleanly shut everything down.

from pyngrok import ngrok
ngrok.kill()
process.terminate()
print('🛑 Ollama server and ngrok tunnel stopped.')